# Multi-Hop and Iterative Retrieval as Search over an Evidence Space

*The bridge in the Information Theory of RAG layer — from the single retrieval step to retrieval as a **search**.*

Context selection was the last topic to treat the candidate pool as fixed; it closed on the line
*"when one passage's answer raises a new question, selection becomes a search over an evolving evidence space."*
This notebook makes that precise, walking the four movements of the topic. Every number is owned by the tested
reference module `multi_hop_iterative_retrieval.py`, which this notebook imports — it never redefines the math.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import multi_hop_iterative_retrieval as mh

c = mh._corpus()
print(f"corpus: {c['K']} companies on the shared finance cloud (seed 7), {c['n_passages']} passages, "
      f"{c['n_chains']} chains (4 one-hop controls, 6 two-hop, 4 three-hop), dim={c['dim']}")

## 1 · The compositional gap: an answer one hop cannot reach

A 2-hop question — *the revenue of the company that acquired Company A's primary supplier* — hides its answer in a
document **unreachable in one retrieval**. On the embedding sphere the answer filing is near-orthogonal to the
query, yet a short reformulation away through a **bridge** filing that names the supplier. We model the bridge as a
mention: company $A$'s filing sits mostly along $u_A$ but carries a $\sin\alpha$ component toward the company $B$
it names, $f_A=\cos\alpha\,u_A+\sin\alpha\,u_B$. Single-hop retrieval cannot reach $B$; the loop can.

In [ ]:
rs = mh.recall_summary(c)
print(f"compositional (>=2 hop) chains:  single-hop answer recall = {rs['comp_single']:.2f}   "
      f"multi-hop answer recall = {rs['comp_multi']:.2f}   (gap = {rs['comp_gap']:.2f})")
print(f"1-hop control:                   single-hop recall = {rs['single_1hop']:.2f}  (directly reachable)")

The three load-bearing cosines (on the exact filing directions): the bridge is retrievable from the query
($q\cdot\text{bridge}>\tau$), the answer is invisible ($q\cdot\text{answer}<\tau$), and **reformulating from
the bridge reaches the answer** — the operator $q'=\text{normalize}(d-\langle d,q\rangle q)$ extracts exactly the
mention direction.

In [ ]:
g = mh.geodesic_cosines(c)
print(f"q . bridge                    = {g['q_dot_bridge']:.3f}   (> tau_hop {g['tau_hop']}: retrievable)")
print(f"q . answer                    = {g['q_dot_answer']:.3f}   (< tau_hop: invisible in one hop)")
print(f"bridge . answer               = {g['bridge_dot_answer']:.3f}   (the mention)")
print(f"reformulate(q,bridge) . answer = {g['reformulated_dot_answer']:.3f}   (reaches the answer)")

## 2 · Multi-hop as search over an evidence space

State = a belief over answer companies; action = a reformulated query; transition = the prereq's retrieve-and-select
step; reward = information. Here is one worked 2-hop trajectory: the belief stays off the answer through the bridge
hop, then snaps onto it at the answer hop.

In [ ]:
ci = mh.worked_chain(c, 2)
traj = mh.multihop_retrieve(c, ci)
print(f"chain companies {c['chains'][ci]['companies']}, truth = company {traj['truth']}, "
      f"hops taken = {traj['hops_taken']}, reached = {traj['reached_answer']}")
print(f"  read passages   : {traj['read_ids']}")
print(f"  belief KL move  : {[round(x,3) for x in traj['kl_moves']]}   (bridge tiny, answer large)")
print(f"  mass on truth   : {[round(x,3) for x in traj['mass_on_truth']]}")

Each hop is the previous topic's **retrieve-and-select** operator, made the transition of a search: from the
$\tau_{\text{hop}}$-reachable pool, greedily select the context by maximizing the imported information-gain oracle
with the imported submodular `greedy_select`.

In [ ]:
sel = mh.greedy_hop_select(c, c['q'][ci], k=2)
print(f"reachable pool size = {len(sel['pool_ids'])}, greedily selected (global ids) = {sel['selected']}")

## 3 · Compounding recall: why hops decay geometrically

End-to-end success is the **product** of per-hop recalls, so it decays geometrically; to hold a target $\rho$ each
hop must over-retrieve to $\rho^{1/k}$. Positive dependence between hops makes the independent product a
**conservative lower bound** (FKG/Harris). We reuse the capstone's cascade machinery so the law is literally the
same one, applied over hops. (Measured per-hop recall is $\approx 1$ on this clean corpus, so the demo uses
illustrative middling retentions.)

In [ ]:
print(f"chain success R1*R2 (0.6, 0.5) = {mh.cascade_recall([mh.DEMO_R1, mh.DEMO_R2]):.3f}, "
      f"over-fetch = {mh.over_fetch_factor([mh.DEMO_R1, mh.DEMO_R2]):.3f}")
print("FKG sweep (positive dependence -> realized recall ABOVE the product):")
for r in mh.dependence_sweep(mh.DEMO_R1, mh.DEMO_R2, [-0.6, 0.0, 0.6]):
    print(f"  rho={r['rho']:+.1f}  R_true={r['R_true']:.3f}  R_indep={r['R_indep']:.3f}  gap={r['gap']:+.3f}")

## 4 · Why greedy cannot shortcut: the supermodular synergy

Information gain is **not submodular** — and the compositional question is its **supermodular** case. The bridge
alone says nothing about the answer; the answer document alone cannot be recognized as relevant; together they
resolve it — the XOR witness context selection used. The operational consequence: single-shot selection **cannot
reach** the answer, so a myopic rule that halts when the belief stops moving would stop at the worthless-looking
bridge. Only reformulating harvests the synergy — and the adaptive hop count equals the chain depth.

In [ ]:
xor = mh.info_gain_xor_witness()
print(f"XOR: answer document alone = {xor['delta_empty']} bits  vs  given the bridge = {xor['delta_given_d1']} bits  "
      f"(supermodular: marginal increases with conditioning)")
sc = mh.singleshot_cannot_reach(c)
print(f"answer in the single-hop pool = {sc['answer_in_single_pool']:.2f}  (single-shot cannot reach)")
print(f"answer after one reformulation (2-hop) = {sc['answer_in_reformulated_pool_2hop']:.2f}")
for n in (1, 2, 3):
    t = mh.multihop_retrieve(c, mh.worked_chain(c, n))
    print(f"  {n}-hop chain -> adaptive hops taken = {t['hops_taken']}")

## Verification

Every pedagogical claim above is an assertion in the reference module; `viz_constants()` prints the numbers the
interactive lab mirrors.

In [ ]:
mh._run_all()

In [ ]:
mh.viz_constants()